In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

df_bronze = spark.read.format("csv").option("header",True).load("/Volumes/scd_prac/bronze/data/customers_day1.csv")

df2 = df_bronze.withColumn("customer_id", col("customer_id").cast(IntegerType()))

df2.write.mode("append").saveAsTable("scd_prac.bronze.data")

display(df2)


In [0]:
%sql

select * from scd_prac.silver.dim_customer


In [0]:
from delta.tables import DeltaTable

df_dest = DeltaTable.forName(spark,"scd_prac.silver.dim_customer")
df_src = spark.read.table("scd_prac.bronze.data")

(
    df_dest.alias("d")
    .merge(
        df_src.alias("s"),
        "d.customer_id = s.customer_id and d.is_current = true"
    )
    .whenMatchedUpdate(
        
        condition="""
            d.customer_name <> s.customer_name
            OR d.city <> s.city
            OR d.status <> s.status
                """,

        set={
            "is_current": "false",
            "effective_end_date": "current_date()"
        }
    )
        .execute()
)

In [0]:
df_dest = DeltaTable.forName(spark,"scd_prac.silver.dim_customer")
df_src = spark.read.table("scd_prac.bronze.data")

(
    df_dest.alias("d").merge(df_src.alias("s"),"d.customer_id = s.customer_id and d.is_current = true")\
        .whenNotMatchedInsert
        (
            values={
            "customer_id": "s.customer_id",
            "customer_name": "s.customer_name",
            "city": "s.city",
            "status": "s.status",
            "effective_start_date": "current_date()",
            "effective_end_date": "null",
            "is_current": "true"
                }
        )
            .execute()
)